In [9]:
from __future__ import annotations

import sys
from copy import deepcopy
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

In [10]:
def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find configs/config.yaml")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.dataset import DatasetSpec, load_dataset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.losses.clip_subspace import (
    compute_clip_subspace_weights,
    compute_clip_raw_similarity_weights,
    compute_random_subspace_weights,
    load_clip_encoder,
    load_dino_encoder,
    WeightedSubset,
)
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.training.unlearn import UnlearnConfig, unlearn
from ebm_unlearning.src.utils.logging import setup_logger
from ebm_unlearning.src.utils.seed import set_seed
from ebm_unlearning.src.utils.tracking import make_tracker

In [11]:
with open(ROOT / "configs" / "config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

set_seed(int(cfg["seed"]))

from datetime import datetime

device = torch.device(cfg.get("device", "cpu"))
logger = setup_logger("unlearn", log_file=str(ROOT / "outputs" / "logs" / "unlearn.log"))
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
tracker = make_tracker(
    "tensorboard",
    log_dir=str(ROOT / "outputs" / "tensorboard" / cfg["data"]["dataset"] / "unlearn" / run_id),
)

# Data: build forget/retain loaders (train only for the update loop)
spec = DatasetSpec(name=cfg["data"]["dataset"], data_dir=str(ROOT / cfg["data"]["data_dir"]), train=True, download=True)
dset = load_dataset(spec)

forget_spec = ForgetSpec(mode=cfg["data"]["forget"]["mode"], class_label=cfg["data"]["forget"]["class_label"])
retain_spec = RetainSpec(mode=cfg["data"]["retain"]["mode"])
forget_all, retain_all = split_forget_retain(dset, forget_spec, retain_spec)

holdout_fraction = float(cfg["evaluation"]["holdout_fraction"])
forget_train, forget_holdout = train_holdout_split(forget_all, holdout_fraction, seed=int(cfg["seed"]))
retain_train, retain_holdout = train_holdout_split(retain_all, holdout_fraction, seed=int(cfg["seed"]) + 1)

batch_size = int(cfg["data"]["batch_size"])
num_workers = int(cfg["data"]["num_workers"])
forget_loader = DataLoader(forget_train, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
retain_loader = DataLoader(retain_train, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)

forget_holdout_loader = DataLoader(forget_holdout, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=True)
retain_holdout_loader = DataLoader(retain_holdout, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=True)


[data] loading cifar100 (train=True, download=True) from /home/owais/machine unlearning/ebm_unlearning/data


In [12]:
# ── CHANGE THESE only when encoder / forget class / k changes ─────────────────
WEIGHTING_MODE  = "pca"    # "pca" | "raw_sim" | "random"
ENCODER_BACKEND = "dino"   # "clip" | "dino"
# ─────────────────────────────────────────────────────────────────────────────

forget_label      = int(cfg["data"]["forget"]["class_label"])
forget_class_name = dset.classes[forget_label]   # works for cifar10, cifar100, mnist
n_components      = int(cfg["unlearning"].get("n_pca_components", 10))

print(f"Dataset      : {cfg['data']['dataset']}")
print(f"Forget class : {forget_label} ({forget_class_name})")
print(f"Encoder      : {ENCODER_BACKEND}  |  Mode: {WEIGHTING_MODE}  |  k={n_components}")

WEIGHT_FN_MAP = {
    "pca":     compute_clip_subspace_weights,
    "raw_sim": compute_clip_raw_similarity_weights,
    "random":  compute_random_subspace_weights,
}

# Load E0 (pretrained reference — frozen)
E0 = EnergyModel(
    in_channels=int(cfg["model"]["in_channels"]),
    hidden_dim=int(cfg["model"]["hidden_dim"]),
    num_classes=int(cfg["model"].get("num_classes", 10)),
    embed_dim=int(cfg["model"].get("embed_dim", 128)),
    backbone=str(cfg["model"].get("backbone", "conv")),
    finetune_stages=int(cfg["model"].get("finetune_stages", 1)),
    imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
)
E0 = load_pretrained(E0, str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)

# Load semantic encoder
print(f"\nLoading {ENCODER_BACKEND.upper()} encoder...")
if ENCODER_BACKEND == "clip":
    enc_model, enc_preprocess, enc_type = load_clip_encoder(device)
else:
    enc_model, enc_preprocess, enc_type = load_dino_encoder(device)
print(f"{ENCODER_BACKEND.upper()} loaded.")

# Compute subspace weights (re-run only when encoder / forget class / k changes)
print(f"\nComputing subspace weights (k={n_components})...")
_retain_weights_cache, _, _ = WEIGHT_FN_MAP[WEIGHTING_MODE](
    model=enc_model,
    preprocess=enc_preprocess,
    raw_data=dset.data,
    forget_indices=forget_train.indices,
    retain_indices=retain_train.indices,
    device=device,
    n_components=n_components,
    encoder_type=enc_type,
)
print("Weights cached. Run next cell to configure lambda and train.")

Dataset      : cifar100
Forget class : 11 (boy)
Encoder      : dino  |  Mode: pca  |  k=30

Loading DINO encoder...
DINO loaded.

Computing subspace weights (k=30)...
  DINO: extracting 400 forget features...
  DINO: extracting 39600 retain features...
  weights — mean=0.0889  max=0.7781  %>0.05: 66.6%
Weights cached. Run next cell to configure lambda and train.


In [13]:
# ── Run this cell whenever lambda_clip / steps / margin changes ───────────────
# Reads fresh config, resets E from E0, rebuilds retain loader from cached weights.
# Does NOT recompute encoder features — runs in seconds.

with open(ROOT / "configs" / "config.yaml") as f:
    cfg = yaml.safe_load(f)

def _ckpt(mode, backend):
    suffix = f"_{backend}" if backend != "clip" else ""
    suffix += "" if mode == "pca" else f"_{mode}"
    return f"outputs/checkpoints/ebm_unlearned_clip_{forget_class_name}{suffix}.pt"

checkpoint_path = str(ROOT / _ckpt(WEIGHTING_MODE, ENCODER_BACKEND))

un_cfg = UnlearnConfig(
    steps=int(cfg["unlearning"]["steps"]),
    lr=float(cfg["unlearning"]["lr"]),
    weight_decay=float(cfg["unlearning"]["weight_decay"]),
    lambda_f=float(cfg["unlearning"]["lambda_f"]),
    lambda_r=float(cfg["unlearning"]["lambda_r"]),
    lambda_m=float(cfg["unlearning"]["lambda_m"]),
    lambda_e=float(cfg["unlearning"]["lambda_e"]),
    lambda_clip=float(cfg["unlearning"].get("lambda_clip", 0.0)),
    n_pca_components=int(cfg["unlearning"].get("n_pca_components", 10)),
    margin=float(cfg["unlearning"]["margin"]),
    log_every=int(cfg["unlearning"]["log_every"]),
    checkpoint_path=checkpoint_path,
)

# Reset E to a fresh copy of pretrained
E = deepcopy(E0)

# Rebuild retain loader from cached weights (no recomputation)
retain_train_weighted = WeightedSubset(retain_train, _retain_weights_cache)
retain_loader = DataLoader(retain_train_weighted, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)

print(f"lambda_clip={un_cfg.lambda_clip}  steps={un_cfg.steps}  k={un_cfg.n_pca_components}")
print(f"Checkpoint → {checkpoint_path}")

lambda_clip=6.0  steps=2000  k=30
Checkpoint → /home/owais/machine unlearning/ebm_unlearning/outputs/checkpoints/ebm_unlearned_clip_boy_dino.pt


In [14]:
E = unlearn(
    E,
    E0,
    forget_loader,
    retain_loader,
    device=device,
    cfg=un_cfg,
    logger=logger,
    tracker=tracker,
    seed=int(cfg["seed"]),
    # holdout loaders omitted — MIA proxy metrics disabled to avoid DataLoader fork issues in Jupyter
)
tracker.close()

[2026-05-31 00:36:56,347] [INFO] [unlearn] step=0 gap_fw=0.2347 total=33.868965 forget=4.832891 retain=2.281976 margin=2.194998 energy_reg=7.731229 clip=0.668931
[2026-05-31 00:37:02,616] [INFO] [unlearn] step=50 gap_fw=3.0015 total=19.065037 forget=2.301785 retain=1.126347 margin=0.614911 energy_reg=19.804428 clip=0.810845
[2026-05-31 00:37:08,623] [INFO] [unlearn] step=100 gap_fw=4.8279 total=11.801823 forget=1.168954 retain=0.581994 margin=0.118937 energy_reg=31.062954 clip=0.777154
[2026-05-31 00:37:14,400] [INFO] [unlearn] step=150 gap_fw=5.8914 total=11.124817 forget=0.467544 retain=0.555843 margin=0.099525 energy_reg=36.585999 clip=0.827122
[2026-05-31 00:37:20,112] [INFO] [unlearn] step=200 gap_fw=6.8742 total=11.294962 forget=0.191108 retain=0.584979 margin=0.033710 energy_reg=40.469509 clip=0.863314
[2026-05-31 00:37:25,919] [INFO] [unlearn] step=250 gap_fw=7.2571 total=11.119392 forget=0.122187 retain=0.533158 margin=0.033154 energy_reg=43.131218 clip=0.931557
[2026-05-31 00